### Setup

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS education.gold_schema")
spark.sql("USE CATALOG education")
spark.sql("USE SCHEMA gold_schema")

DataFrame[]

### Read Enriched Data

In [0]:
from pyspark.sql.functions import *

df = spark.table("education.silver_schema.student_enriched")

### KPI 1: Course Performance Summary

In [0]:
course_perf = df.groupBy("code_module") \
    .agg(
        count("*").alias("total_students"),
        avg("score").alias("avg_score"),
        sum(when(col("final_result") == "Pass", 1).otherwise(0)).alias("passes"),
        sum(when(col("final_result") == "Withdrawn", 1).otherwise(0)).alias("dropouts")
    )

course_perf.write.format("delta").mode("overwrite") \
    .saveAsTable("education.gold_schema.course_performance")

### Gold Layer: Course Performance Analytics

In [0]:
display(spark.table("education.gold_schema.course_performance"))

code_module,total_students,avg_score,passes,dropouts
FFF,61778,77.11556217423679,37101,6860
GGG,15926,79.52924485716129,9644,509
AAA,3404,69.04752004752005,2468,380
EEE,13044,76.32463164638052,7788,1366
DDD,38613,69.94089222308247,20529,6814
BBB,46916,76.506310056237,26991,5347
CCC,33485,74.79321559925833,13506,9346


### KPI 2: Pass Rate

In [0]:
pass_rate = df.groupBy("code_module") \
    .agg(
        (sum(when(col("final_result") == "Pass", 1).otherwise(0)) /
         count("*") * 100).alias("pass_percentage")
    )

pass_rate.write.format("delta").mode("overwrite") \
    .saveAsTable("education.gold_schema.pass_rate")

### Pass Rate Insights for Decision Making

In [0]:
display(spark.table("education.gold_schema.pass_rate"))

code_module,pass_percentage
FFF,60.055359513095276
GGG,60.555067185734025
AAA,72.50293772032902
EEE,59.70561177552898
DDD,53.16603216533292
BBB,57.5304800068207
CCC,40.334478124533376


### KPI 3: Dropout Rate

In [0]:
dropout_rate = df.groupBy("code_module") \
    .agg(
        (sum(when(col("final_result") == "Withdrawn", 1).otherwise(0)) /
         count("*") * 100).alias("dropout_percentage")
    )

dropout_rate.write.format("delta").mode("overwrite") \
    .saveAsTable("education.gold_schema.dropout_rate")

### Student Dropout Rate Summary

In [0]:
display(spark.table("education.gold_schema.dropout_rate"))

code_module,dropout_percentage
FFF,11.104276603321571
GGG,3.1960316463644354
AAA,11.163337250293772
EEE,10.472247776755596
DDD,17.646906482272808
BBB,11.39696478813198
CCC,27.911004927579512


### KPI 4: Risk Distribution

In [0]:
risk_dist = df.groupBy("risk_flag") \
    .agg(count("*").alias("count"))

risk_dist.write.format("delta").mode("overwrite").saveAsTable("education.gold_schema.risk_distribution")

### Student Risk Distribution

In [0]:
display(spark.table("education.gold_schema.risk_distribution"))

risk_flag,count
High,64795
Medium,14459
Low,133912


### KPI 5: Top Students

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, row_number, col

window_spec = Window.partitionBy("code_module").orderBy(desc("score"))

top_students = df.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 5)

top_students.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("education.gold_schema.top_students")

### Top Students Data

In [0]:
display(spark.table("education.gold_schema.top_students"))

code_module,code_presentation,id_assessment,id_student,final_result,num_of_prev_attempts,date_registration,date_unregistration,score,weight,module_presentation_length,enrollment_duration,risk_flag,weighted_score,rank
AAA,2013J,1756,2536991,Distinction,0,-87,0,98,30.0,268,87,Low,29.4,1
AAA,2013J,1759,342007,Withdrawn,0,-71,51,95,20.0,268,122,High,19.0,2
AAA,2014J,34882,2596621,Pass,0,-101,0,95,0.0,269,101,Low,0.0,3
AAA,2014J,1762,527100,Distinction,0,-24,0,95,30.0,269,24,Low,28.5,4
AAA,2013J,1753,296332,Distinction,0,-115,0,95,20.0,268,115,Low,19.0,5
BBB,2013B,14991,354058,Fail,0,-29,0,100,1.0,240,29,High,1.0,1
BBB,2013B,14995,271036,Pass,3,-9,0,100,1.0,240,9,Medium,1.0,2
BBB,2013B,14995,352364,Distinction,0,-30,0,100,1.0,240,30,Low,1.0,3
BBB,2013B,14995,120994,Fail,0,-19,0,100,1.0,240,19,High,1.0,4
BBB,2013B,14994,388360,Fail,0,-122,0,100,1.0,240,122,High,1.0,5


### # **SQL BUSINESS QUERIES**

### 1. Top 5 courses by pass rate

In [0]:
%sql
SELECT * 
FROM education.gold_schema.pass_rate
ORDER BY pass_percentage DESC
LIMIT 5;

code_module,pass_percentage
AAA,72.50293772032902
GGG,60.555067185734025
FFF,60.055359513095276
EEE,59.70561177552898
BBB,57.5304800068207


### 2. Courses with highest dropout

In [0]:
%sql
SELECT * 
FROM education.gold_schema.dropout_rate
ORDER BY dropout_percentage DESC;

code_module,dropout_percentage
CCC,27.911004927579512
DDD,17.646906482272808
BBB,11.39696478813198
AAA,11.163337250293772
FFF,11.104276603321571
EEE,10.472247776755596
GGG,3.1960316463644354


### 3. Average score per course

In [0]:
%sql
SELECT code_module, AVG(score) AS avg_score
FROM education.silver_schema.student_enriched
GROUP BY code_module;

code_module,avg_score
AAA,69.04752004752005
BBB,76.506310056237
CCC,74.79321559925833
DDD,69.94089222308247
EEE,76.32463164638052
FFF,77.11556217423679
GGG,79.52924485716129


### 4. Count of students per result

In [0]:
%sql
SELECT final_result, COUNT(*) 
FROM education.silver_schema.student_enriched
GROUP BY final_result;

final_result,COUNT(*)
Pass,118027
Fail,34173
Withdrawn,30622
Distinction,30344


### 5. Top 10 students overall

In [0]:
%sql
SELECT id_student, score
FROM education.silver_schema.student_enriched
ORDER BY score DESC
LIMIT 10;

id_student,score
390442,100
480458,100
354058,100
500791,100
388360,100
271036,100
352364,100
120994,100
486099,100
505961,100


### 6. At-risk students list

In [0]:
%sql
SELECT *
FROM education.gold_schema.at_risk_students;

code_module,code_presentation,id_assessment,id_student,final_result,date_registration,date_unregistration,score,weight,module_presentation_length,enrollment_duration,risk_flag,weighted_score
AAA,2013J,1754,268733,Withdrawn,-17,187,35,20.0,268,204,At Risk,7.0
AAA,2013J,1756,392756,Fail,-58,0,38,30.0,268,null,At Risk,11.4
AAA,2014J,1759,468120,Fail,-106,0,38,20.0,269,null,At Risk,7.6
BBB,2013B,25349,371679,Pass,-42,0,26,12.5,240,null,At Risk,3.25
BBB,2013B,14987,549204,Fail,-16,0,37,18.0,240,null,At Risk,6.66
BBB,2013B,14984,549713,Withdrawn,-29,57,0,5.0,240,86,At Risk,0.0
BBB,2013B,14995,559744,Pass,-21,0,20,1.0,240,null,At Risk,0.2
BBB,2013J,15004,68556,Fail,-23,0,20,1.0,268,null,At Risk,0.2
BBB,2013J,14999,510868,Withdrawn,-29,214,30,18.0,268,243,At Risk,5.4
BBB,2013J,15020,585074,Fail,-74,0,1,0.0,268,null,At Risk,0.0


### 7. Students with multiple attempts

In [0]:
%sql
SELECT *
FROM education.silver_schema.student_enriched
WHERE num_of_prev_attempts > 0;

code_module,code_presentation,id_assessment,id_student,final_result,num_of_prev_attempts,date_registration,date_unregistration,score,weight,module_presentation_length,enrollment_duration,risk_flag,weighted_score
AAA,2014J,1762,121349,Pass,1,-28,0,86,30.0,269,28,Medium,25.8
AAA,2014J,1760,129955,Withdrawn,1,-143,143,76,20.0,269,286,High,15.2
AAA,2014J,1762,147756,Pass,1,-41,0,68,30.0,269,41,Medium,20.4
AAA,2014J,1762,155984,Pass,1,-156,0,83,30.0,269,156,Medium,24.9
AAA,2014J,1752,2429854,Fail,1,-37,0,51,10.0,269,37,High,5.1
BBB,2013B,14993,152153,Withdrawn,1,-93,157,80,1.0,240,250,High,0.8
BBB,2013B,14995,185349,Pass,2,-54,0,80,1.0,240,54,Medium,0.8
BBB,2013B,14995,271036,Pass,3,-9,0,100,1.0,240,9,Medium,1.0
BBB,2013B,14995,351290,Pass,2,-54,0,80,1.0,240,54,Medium,0.8
BBB,2013B,25349,371679,Pass,2,-42,0,26,12.5,240,42,Medium,3.25


### 8. Course with highest avg duration

In [0]:
%sql
SELECT code_module, AVG(enrollment_duration) AS avg_duration
FROM education.silver_schema.student_enriched
GROUP BY code_module
ORDER BY avg_duration DESC;

### 9. Pass vs Fail comparison

In [0]:
%sql
SELECT code_module,
SUM(CASE WHEN final_result='Pass' THEN 1 ELSE 0 END) AS pass,
SUM(CASE WHEN final_result='Fail' THEN 1 ELSE 0 END) AS fail
FROM education.silver_schema.student_enriched
GROUP BY code_module;

code_module,pass,fail
AAA,2468,335
BBB,26991,8280
CCC,13506,5108
DDD,20529,7464
EEE,7788,1514
FFF,37101,9170
GGG,9644,2302


### 10. Risk distribution

In [0]:
%sql
SELECT risk_flag, COUNT(*)
FROM education.silver_schema.student_enriched
GROUP BY risk_flag;

risk_flag,COUNT(*)
Low,133912
High,64795
Medium,14459
